# RLHF: 인간 피드백 기반 강화학습 - 실습 코드 2: RLHF 전체 파이프라인 구현 (SFT → RM → PPO)

- Tutorial ID: `expand-rlhf`
- Tutorial: RLHF: 인간 피드백 기반 강화학습
- Section ID: `expand-rlhf-code-2`
- Section: 실습 코드 2: RLHF 전체 파이프라인 구현 (SFT → RM → PPO)


In [ ]:
# ============================================================
# 코드 읽는 법 — 실습 코드 2: RLHF 전체 파이프라인 구현 (SFT → RM → PPO)
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) 선호쌍 chosen/rejected가 loss와 policy update 신호로 바뀌는 흐름 확인
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ── Step 1: SFT (Supervised Fine-Tuning) ──
def train_sft(model, tokenizer, sft_data, epochs=3, lr=2e-5):
    """인간이 작성한 고품질 응답으로 지도학습"""
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    model.train()
    
        for epoch in range(epochs):
                total_loss = 0
                for prompt, response in sft_data:
            # 프롬프트 + 응답을 하나의 시퀀스로
            full_text = f"User: {prompt}\nAssistant: {response}"
            input_ids = tokenizer.encode(full_text, return_tensors="pt")
            
            # 응답 부분만 loss 계산 (프롬프트 마스킹)
            prompt_len = len(tokenizer.encode(f"User: {prompt}\nAssistant:"))
            labels = input_ids.clone()
            labels[:, :prompt_len] = -100  # 프롬프트 부분 무시
            
            outputs = model(input_ids, labels=labels)
                        loss = outputs.loss
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
        
                print(f"SFT Epoch {epoch+1}: loss={total_loss/len(sft_data):.4f}")
    
    return model

# ── Step 2: Reward Model 학습 (Bradley-Terry) ──
class RewardModel(nn.Module):
    """응답 품질을 스칼라 점수로 평가하는 보상 모델"""
        def __init__(self, base_model, hidden_dim=768):
        super().__init__()
        self.base = base_model
        # 마지막 레이어 위에 보상 헤드 추가
        self.reward_head = nn.Sequential(
            nn.Linear(hidden_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 1),
        )
    
        def forward(self, input_ids):
        """입력 시퀀스의 보상 점수 계산"""
        with torch.no_grad():
            hidden = self.base(input_ids).last_hidden_state
        # 마지막 토큰의 은닉 상태 사용
        last_hidden = hidden[:, -1, :]
        reward = self.reward_head(last_hidden)
        return reward.squeeze(-1)

def train_reward_model(rm, rm_data, epochs=2, lr=1e-5):
    """Bradley-Terry 모델로 보상 모델 학습
    
    Loss = -log σ(r(x, y_w) - r(x, y_l))
    """
    optimizer = torch.optim.AdamW(rm.parameters(), lr=lr)
    rm.train()
    
        for epoch in range(epochs):
                total_loss = 0
        correct = 0
        total = 0
        
                for prompt, chosen, rejected in rm_data:
            chosen_ids = torch.tensor([chosen]).long()
            rejected_ids = torch.tensor([rejected]).long()
            
            r_chosen = rm(chosen_ids)
            r_rejected = rm(rejected_ids)
            
            # Bradley-Terry Loss
                        loss = -F.logsigmoid(r_chosen - r_rejected)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            correct += (r_chosen > r_rejected).float().item()
            total += 1
        
        acc = correct / total * 100
                print(f"RM Epoch {epoch+1}: loss={total_loss/len(rm_data):.4f}, acc={acc:.1f}%")
    
    return rm

# ── Step 3: PPO (Proximal Policy Optimization) ──
def ppo_train_step(
    policy_model,    # 학습할 RL 정책
    ref_model,       # 참조 모델 (SFT 모델)
    reward_model,    # 보상 모델
    prompt,          # 입력 프롬프트
    tokenizer,
    beta=0.1,        # KL 페널티 계수
    epsilon=0.2,     # PPO 클리핑
):
    """PPO 한 스텝 학습"""
    # 1. 정책으로 응답 생성
    input_ids = tokenizer.encode(prompt, return_tensors="pt")
    response = policy_model.generate(input_ids, max_new_tokens=128, do_sample=True)
    
    # 2. 보상 계산
    reward = reward_model(response)
    
    # 3. KL 페널티 계산 (참조 모델에서 너무 멀어지지 않도록)
    with torch.no_grad():
        log_prob_policy = -policy_model(response, labels=response).loss
        log_prob_ref = -ref_model(response, labels=response).loss
    kl_penalty = beta * (log_prob_policy - log_prob_ref)
    
    # 4. 최종 보상 = 보상 - KL 페널티
    final_reward = reward - kl_penalty
    
    # 5. PPO 클리핑 목적 함수
    ratio = torch.exp(log_prob_policy - log_prob_ref)  # π_new / π_old
    clipped_ratio = torch.clamp(ratio, 1 - epsilon, 1 + epsilon)
    
    # Advantage (단순화: final_reward를 직접 사용)
    advantage = final_reward
    
        ppo_loss = -torch.min(
        ratio * advantage,
        clipped_ratio * advantage
    ).mean()
    
    return ppo_loss, final_reward.item()

# ── 전체 RLHF 파이프라인 실행 예시 ──
print("=== RLHF 전체 파이프라인 ===\n")
print("Step 1: SFT (Supervised Fine-Tuning)")
print("  → 인간 작성 응답으로 기본 대화 능력 학습\n")

print("Step 2: Reward Model 학습")
print("  → 인간 선호도 비교 데이터로 보상 모델 학습\n")

print("Step 3: PPO 최적화")
print("  → 보상 최대화 + KL 페널티로 정책 미세 조정\n")

print("수학적 핵심:")
print("  SFT Loss:  -log π(y|x)")
print("  RM Loss:   -log σ(r(x,y_w) - r(x,y_l))")
print("  PPO:       max E[r(x,y) - β·KL(π_θ || π_ref)]")